# Crash Exploratory Analysis using SQL and Python

## Imports & Setting up the working enviroment

In [27]:
import duckdb
import pandas as pd
pd.set_option('display.max_rows', 100)

In [22]:
con = duckdb.connect('crashes.duckdb')

After establised the connection we can start writing SQL code!

## Load the data

In [23]:
con.execute("""
    CREATE OR REPLACE TABLE crashes AS
    SELECT *
    FROM read_csv_auto('data/Crash_Analysis_System__CAS__data.csv', header=true, sample_size=-1)
""")

result = con.execute("SELECT COUNT(*) AS total_crashes FROM crashes").fetchdf()
print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_crashes
0         913464


In [28]:
con.execute(""" DESCRIBE crashes """).fetchdf()

,column_name,column_type,null,key,default,extra
0,X,DOUBLE,YES,None,None,None
1,Y,DOUBLE,YES,None,None,None
2,OBJECTID,BIGINT,YES,None,None,None
3,advisorySpeed,BIGINT,YES,None,None,None
4,areaUnitID,BIGINT,YES,None,None,None
5,bicycle,BIGINT,YES,None,None,None
6,bridge,BIGINT,YES,None,None,None
7,bus,BIGINT,YES,None,None,None
8,carStationWagon,BIGINT,YES,None,None,None
9,cliffBank,BIGINT,YES,None,None,None


There are 913,464 rows and 72 columns in the table.

## Data profiling

In [25]:
con.execute("""
    SELECT
        *
    FROM crashes
    LIMIT 10""").fetchdf()     

,X,Y,OBJECTID,advisorySpeed,areaUnitID,bicycle,bridge,bus,carStationWagon,cliffBank,...,train,tree,truck,unknownVehicleType,urban,vanOrUtility,vehicle,waterRiver,weatherA,weatherB
0,1747565.0,5427642.0,1,<NA>,575300,0,<NA>,0,2,<NA>,...,<NA>,<NA>,0,0,Urban,0,<NA>,<NA>,Fine,Null
1,1544693.0,5170620.0,2,<NA>,587020,0,0,0,1,0,...,0,0,0,0,Open,0,0,0,Fine,Null
2,1615685.0,5423963.0,3,<NA>,584201,0,0,0,0,0,...,0,0,0,0,Urban,0,0,0,Fine,Null
3,1799588.0,5821916.0,4,<NA>,527007,0,<NA>,0,2,<NA>,...,<NA>,<NA>,0,0,Urban,0,<NA>,<NA>,Fine,Null
4,1570549.0,5176071.0,5,<NA>,591101,1,<NA>,0,0,<NA>,...,<NA>,<NA>,0,0,Urban,0,<NA>,<NA>,Fine,Null
5,1680069.0,5404460.0,6,<NA>,581230,0,<NA>,0,3,<NA>,...,<NA>,<NA>,0,0,Urban,0,<NA>,<NA>,Fine,Null
6,1394487.0,4916691.0,7,<NA>,606100,0,0,0,2,0,...,0,0,0,0,Urban,0,0,0,Fine,Null
7,1789692.0,5858082.0,8,<NA>,526900,0,0,0,1,0,...,0,0,0,0,Urban,1,0,0,Fine,Null
8,1623513.0,5430552.0,9,<NA>,582700,0,<NA>,0,3,<NA>,...,<NA>,<NA>,0,0,Urban,0,<NA>,<NA>,Light rain,Null
9,1405988.0,4916533.0,10,<NA>,603000,0,<NA>,0,2,<NA>,...,<NA>,<NA>,0,0,Urban,0,<NA>,<NA>,Light rain,Null


The dataset is denormalised into a single flat table which is suitable for the purpose of this exploratory analysis.

What year this dataset covered?

In [26]:
con.execute("""
    SELECT
        MIN(crashYear) AS earliest_year
        , MAX(crashYear) AS latest_year
        , COUNT(distinct crashYear) AS total_years
    FROM crashes
    """).fetchdf()

,earliest_year,latest_year,total_years
0,2000,2026,27


The dataset runs from 2000 to 2026 (27 years).

We nocticed there are many columns contains "<N/A>" values - Pandas missing values display as well as the text "Null" which is the dataset default Null. Let's see how many NULLS are there in each columns.

In [67]:
column_names = con.execute("""DESCRIBE crashes""").fetchdf()['column_name'].tolist()

select_parts = ", ".join([
    f'COUNT(*) FILTER (WHERE "{col}" IS NULL OR CAST("{col}" AS VARCHAR) = \'Null\') AS "{col}"'
    for col in column_names
])

In [68]:
result = con.execute(f"SELECT {select_parts} FROM crashes").fetchdf()

In [70]:
null_df = result.T.reset_index()
null_df.columns = ['column_name', 'null_count']
null_df =null_df.sort_values(by='null_count', ascending=False)

null_df['pct_null'] = null_df['null_count'] / 913464 * 100

In [75]:
null_df.head(5)

,column_name,null_count,pct_null
27,intersection,913464,100.000000
14,crashRoadSideRoad,913464,100.000000
56,temporarySpeedLimit,897788,98.283895
40,pedestrian,883236,96.690838
71,weatherB,881993,96.554763


In [72]:
null_df.tail(5)

,column_name,null_count,pct_null
17,crashYear,0,0.0
15,crashSeverity,0,0.0
44,roadCharacter,0,0.0
59,trafficControl,0,0.0
66,urban,0,0.0


There are columns where we have complete data such as crashYear, crashSeverity, roadCharacter etc but there are also columns where we have absolutely all null values such as intersection, crashRoadSideRoad etc. Additionally there are 29 columns with more than 50% nulls.

Any duplicated records?

In [76]:
con.execute("""SELECT COUNT(DISTINCT OBJECTID) FROM crashes""").fetchdf()

,count(DISTINCT OBJECTID)
0,913464


There are no duplicated keys, all records are distinct.

#### Data Profile Summary
##### Dataset Overview
The Crash Analysis System (CAS) dataset contains **913,464 crash records** spanning **27 years** (2000-2026), organized in a single denormalized table with **72 columns**. Each record represents a unique crash incident identified by the OBJECTID field.

##### Data Quality Assessment

**Completeness:**
- **High-quality columns**: Several key fields have complete data including crashYear, crashSeverity, and roadCharacter
- **Severely incomplete columns**: Multiple columns contain 100% null values (intersection, crashRoadSideRoad, etc.)
- **Moderate missingness**: 29 columns have more than 50% missing data

**Data Integrity:**
- ✅ **No duplicates**: All 913,464 records have unique OBJECTID values
- ⚠️ **Mixed null representations**: Dataset uses both actual NULL values and text "Null" as missing data indicators

##### Key Data Characteristics
- **Temporal coverage**: 27-year span provides extensive historical perspective for trend analysis
- **Structure**: Flat table design suitable for exploratory analysis and reporting
- **Scale**: Nearly 1 million records provide substantial sample size for statistical analysis